# Attention fusion with Qwen captions

This notebook trains one experiment per **split seed**:

1. **Splits** — Read `train.csv` / `val.csv` / `test.csv` from `data/splits/seed_<id>/` (same layout as the image-only CSV protocol).
2. **Images** — Resolve `image_path` from those CSVs to files under the repo (`dataset/` or `data/raw dataset/` fallback).
3. **Text** — Look up **Qwen** captions in `data/captions/qwen25vl_caption_full.csv` by `image_path` (success rows only when `status` is present). BERT encodes that string; CLIP encodes the image.
4. **Vision encoder** — For each seed, load **fine-tuned CLIP ViT-B/32** weights from `FASHION_CLIP_FINETUNED_MODELS_DIR` (default: `experiments/imageonly_clip_finetuned_robustness/models/seed_<id>_best_model.pth`). CLIP and BERT stay frozen; only the **attention fusion block** and **MLP classifier** are trained.
5. **Seeds** — All `seed_*` directories under `data/splits/` are discovered automatically. Set `RUN_ALL_SEEDS` / `SMOKE_SEED` in the first code cell.

**Reproducibility:** After editing notebook cells, use **Kernel → Restart** and **Run All** so `ATTN_FUSION_DATASET_CELL_VERSION` matches the dataset definition.

**Artifacts:** Metrics and fusion checkpoints are written under `experiments/attention_fusion_finetuned_clip_qwen_v2/`.

In [1]:
import os
import re
import json
import shutil
import subprocess
import time
import warnings
from datetime import datetime
from pathlib import Path
from urllib.parse import unquote

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import matplotlib.pyplot as plt

os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

# --- project root ---
_walk = os.path.abspath(os.getcwd())
for _ in range(10):
    if os.path.isdir(os.path.join(_walk, "experiments")) and os.path.isdir(os.path.join(_walk, "data")):
        PROJECT_ROOT = _walk
        break
    _walk = os.path.dirname(_walk)
else:
    PROJECT_ROOT = os.path.abspath(os.getcwd())
os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)

# --- toggles ---
RUN_ALL_SEEDS = True   # set True for full 30-seed loop
SMOKE_SEEDS = [13, 14, 16, 17, 45]        # used when RUN_ALL_SEEDS is False

# --- training hyperparameters ---
LEARNING_RATE = 5e-5
BATCH_SIZE = 32
EARLY_STOPPING_PATIENCE = 5
DROPOUT = 0.5
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 20
MODEL_INIT_SEED = 42
NUM_WORKERS = 0


def pick_training_device():
    # Prefer NVIDIA CUDA, then Apple MPS, then CPU.
    if torch.cuda.is_available():
        return torch.device("cuda")
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def print_torch_compute_diagnostics():
    print("torch:", torch.__version__)
    vcuda = getattr(torch.version, "cuda", None)
    print("torch.version.cuda (wheel build):", vcuda)
    print("torch.cuda.is_available():", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA device 0:", torch.cuda.get_device_name(0))
        return
    if vcuda:
        print(
            "This PyTorch build includes CUDA, but the GPU is not usable from this process. "
            "Typical causes: NVIDIA driver/kernel vs user-space library mismatch (often fixed by a reboot), "
            "no discrete GPU, VM/WSL without GPU passthrough, or a driver/GPU pairing that rejects this CUDA build."
        )
    nv = shutil.which("nvidia-smi")
    if nv:
        try:
            r = subprocess.run([nv, "-L"], capture_output=True, text=True, timeout=10)
            msg = ((r.stdout or "") + (r.stderr or "")).strip()
            if msg:
                print("nvidia-smi -L:\n", msg)
            low = msg.lower()
            if "version mismatch" in low or "failed to initialize nvml" in low:
                print(
                    "\n>>> Driver fix: reboot so the loaded NVIDIA kernel module matches libnvidia-ml.so, "
                    "or reinstall/repair the NVIDIA driver until `nvidia-smi` works outside Jupyter."
                )
        except Exception as ex:
            print("nvidia-smi check failed:", ex)
    else:
        print("nvidia-smi not on PATH (install NVIDIA driver + tools, or use a CUDA-capable machine).")


device = pick_training_device()
print_torch_compute_diagnostics()
print("Selected device:", device)

CAPTION_CSV = os.path.join(PROJECT_ROOT, "data", "captions", "qwen25vl_caption_full.csv")
if not os.path.isfile(CAPTION_CSV):
    raise FileNotFoundError(
        f"Caption CSV missing: {CAPTION_CSV}\n"
        "Ensure `git pull` brought in data/captions/qwen25vl_caption_full.csv. "
        "If the first cell still mentions qwen25vl_prompt_a_v2_full_captions.csv, reload this notebook "
        "from disk (discard in-memory edits), then Kernel → Restart and Run All."
    )

CLIP_FINETUNED_ROOT = os.environ.get(
    "FASHION_CLIP_FINETUNED_MODELS_DIR",
    os.path.join(PROJECT_ROOT, "experiments", "imageonly_clip_finetuned_robustness", "models"),
)

SPLITS_ROOT = os.path.join(PROJECT_ROOT, "data", "splits")
EXPERIMENT_ROOT = os.path.join(PROJECT_ROOT, "experiments", "attention_fusion_finetuned_clip_qwen_v2")


def discover_split_seeds(splits_root: str):
    seeds = []
    for name in os.listdir(splits_root):
        m = re.match(r"seed_(\d+)$", name)
        if m and os.path.isdir(os.path.join(splits_root, name)):
            seeds.append(int(m.group(1)))
    return sorted(seeds)


SEEDS = discover_split_seeds(SPLITS_ROOT)
_dfq = pd.read_csv(CAPTION_CSV)
if "style" in _dfq.columns:
    all_styles = sorted(_dfq["style"].dropna().astype(str).unique())
else:
    raise ValueError("Qwen CSV must have 'style' for num_classes")
style_to_idx = {s: i for i, s in enumerate(all_styles)}
num_classes = len(all_styles)

METRICS_DIR = os.path.join(EXPERIMENT_ROOT, "metrics")
ARTIFACTS_DIR = os.path.join(EXPERIMENT_ROOT, "artifacts")
for d in [
    METRICS_DIR,
    os.path.join(METRICS_DIR, "experiments"),
    os.path.join(ARTIFACTS_DIR, "models"),
    os.path.join(ARTIFACTS_DIR, "learning_curves"),
    os.path.join(ARTIFACTS_DIR, "comparison_plots"),
]:
    os.makedirs(d, exist_ok=True)

print("CAPTION_CSV:", CAPTION_CSV)
print("CLIP_FINETUNED_ROOT:", CLIP_FINETUNED_ROOT)
print("EXPERIMENT_ROOT:", EXPERIMENT_ROOT)
print("num_classes:", num_classes)
print("SEEDS count:", len(SEEDS), "| first few:", SEEDS[:5])

with open(os.path.join(METRICS_DIR, "seeds_list.txt"), "w") as f:
    f.write("Seeds from data/splits/seed_*\n")
    for s in SEEDS:
        f.write(f"{s}\n")


def resolve_split_image_path(row_image_path, base_dir):
    # Map split CSV image_path to on-disk file (same rules as ImageOnly_CLIP_Finetune).
    base = base_dir or "."
    p = str(row_image_path)
    if not os.path.isabs(p):
        rel = p.replace(chr(92), "/")
        dataset_top = os.path.join(base, "dataset")
        if rel.startswith("dataset/") and not os.path.isdir(dataset_top):
            p = os.path.join(base, "data", "raw dataset", rel[len("dataset/") :])
        else:
            p = os.path.join(base, p)
    if "%" in p:
        parts = p.replace(chr(92), "/").split("/")
        p = "/".join(unquote(part) if "%" in part else part for part in parts)
    return os.path.normpath(p)


def load_qwen_captions(csv_path: str, base_dir: str):
    # Map image_path -> Qwen caption (success rows only). Register raw, normpath, naive join, and resolve_split_image_path.
    df = pd.read_csv(csv_path)
    if "status" in df.columns:
        df = df[df["status"].astype(str).str.lower() == "success"].copy()
    if "caption" not in df.columns:
        raise ValueError("Expected column 'caption' in caption CSV")

    def register(keys_dict, raw_path, caption_text):
        cap = caption_text.strip()
        p = str(raw_path)
        keys_dict[p] = cap
        keys_dict[os.path.normpath(p)] = cap
        if base_dir and not os.path.isabs(p):
            abs_p = os.path.normpath(os.path.join(base_dir, p))
            keys_dict[abs_p] = cap
        res = resolve_split_image_path(p, base_dir or ".")
        keys_dict[res] = cap

    d = {}
    for _, row in df.iterrows():
        c = row["caption"]
        if not isinstance(c, str) or not c.strip():
            continue
        register(d, row["image_path"], c)
    return d


BASE_DIR = PROJECT_ROOT
captions_dict = load_qwen_captions(CAPTION_CSV, BASE_DIR)
print("Qwen caption dict size (keys may include path variants):", len(captions_dict))


def print_fusion_path_audit(smoke_seed: int):
    print("=== Path audit (required files / dirs) ===")
    img_ok = os.path.isdir(os.path.join(PROJECT_ROOT, "dataset")) or os.path.isdir(
        os.path.join(PROJECT_ROOT, "data", "raw dataset")
    )
    items = [
        ("CAPTION_CSV", os.path.isfile(CAPTION_CSV)),
        (
            "repo dataset/ OR data/raw dataset/",
            img_ok,
        ),
        ("SPLITS_ROOT", os.path.isdir(SPLITS_ROOT)),
        ("CLIP_FINETUNED_ROOT", os.path.isdir(CLIP_FINETUNED_ROOT)),
    ]
    for label, ok in items:
        print(f"  [{'x' if ok else ' '}] {label}")
    if not all(ok for _, ok in items):
        raise RuntimeError("Path audit failed — fix paths above.")

    train_csv = os.path.join(SPLITS_ROOT, f"seed_{smoke_seed}", "train.csv")
    if not os.path.isfile(train_csv):
        raise FileNotFoundError(f"Missing split train.csv: {train_csv}")
    tr = pd.read_csv(train_csv, nrows=5)
    ok_files = 0
    for i, row in tr.iterrows():
        raw = str(row["image_path"])
        res = resolve_split_image_path(raw, BASE_DIR)
        ex = os.path.isfile(res)
        ok_files += int(ex)
        cap = any(
            k in captions_dict for k in (res, raw, os.path.normpath(str(raw)))
        )
        print(f"  [{'x' if ex else ' '}] smoke row {i} image on disk")
        print(f"      [{'x' if cap else ' '}] Qwen caption key")
        print(f"      raw={raw}")
        print(f"      resolved={res}")
    if ok_files == 0:
        raise RuntimeError("Path audit: sample rows resolve to no image files.")

    ck = os.path.join(CLIP_FINETUNED_ROOT, f"seed_{smoke_seed}_best_model.pth")
    if not os.path.isfile(ck):
        raise FileNotFoundError(f"Missing CLIP checkpoint for smoke seed: {ck}")
    print(f"  [x] smoke CLIP ckpt seed_{smoke_seed}_best_model.pth")
    print("=== Path audit OK ===")


print_fusion_path_audit(SMOKE_SEEDS[0])


PROJECT_ROOT: /home/ding-zhang/Dongmei/DATA255/Project
torch: 2.1.2+cu121
torch.version.cuda (wheel build): 12.1
torch.cuda.is_available(): True
CUDA device 0: NVIDIA GeForce RTX 4090
Selected device: cuda
CAPTION_CSV: /home/ding-zhang/Dongmei/DATA255/Project/data/captions/qwen25vl_caption_full.csv
CLIP_FINETUNED_ROOT: /home/ding-zhang/Dongmei/DATA255/Project/experiments/imageonly_clip_finetuned_robustness/models
EXPERIMENT_ROOT: /home/ding-zhang/Dongmei/DATA255/Project/experiments/attention_fusion_finetuned_clip_qwen_v2
num_classes: 14
SEEDS count: 30 | first few: [13, 14, 16, 17, 45]
Qwen caption dict size (keys may include path variants): 39588
=== Path audit (required files / dirs) ===
  [x] CAPTION_CSV
  [x] repo dataset/ OR data/raw dataset/
  [x] SPLITS_ROOT
  [x] CLIP_FINETUNED_ROOT
  [x] smoke row 0 image on disk
      [x] Qwen caption key
      raw=dataset/mode/9234303ea1b3437acd91437fb29df533.jpg
      resolved=/home/ding-zhang/Dongmei/DATA255/Project/data/raw dataset/mode/9

## Load CLIP + BERT

In [2]:
import clip
from transformers import AutoTokenizer, AutoModel

print("Loading CLIP ViT-B/32 …")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device, jit=False)
clip_model = clip_model.float()
clip_model.eval()

print("Loading BERT …")
fashionbert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
fashionbert_model = AutoModel.from_pretrained("bert-base-uncased").to(device)
fashionbert_model.eval()


def load_finetuned_clip_weights(clip_model, ckpt_path, map_location):
    # Load partial fine-tuned CLIP from ImageOnlyFashionClassifier checkpoint.
    ckpt_path = Path(ckpt_path)
    if not ckpt_path.is_file():
        raise FileNotFoundError(f"Missing checkpoint: {ckpt_path}")
    sd = torch.load(ckpt_path, map_location=map_location)
    clip_sd = {k[len("clip_model.") :]: v for k, v in sd.items() if k.startswith("clip_model.")}
    if len(clip_sd) == 0:
        raise ValueError(f"No clip_model.* keys in {ckpt_path} (wrong checkpoint format?)")
    clip_model.load_state_dict(clip_sd, strict=True)
    clip_model.eval()
    for p in clip_model.parameters():
        p.requires_grad = False
    print("  Loaded fine-tuned CLIP:", ckpt_path.name, "|", len(clip_sd), "tensor keys")


print("✅ Encoders ready (reload CLIP weights per seed before training).")


Loading CLIP ViT-B/32 …
Loading BERT …


2026-05-23 21:26:42.211469: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-23 21:26:42.232545: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


✅ Encoders ready (reload CLIP weights per seed before training).


2026-05-23 21:26:42.906155: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Dataset + attention fusion model

In [3]:
class FashionMultiModalDataset(Dataset):
    # Split CSV rows + Qwen captions + CLIP preprocess. Excludes rows without file or caption.

    def _resolve_path(self, row_image_path):
        return resolve_split_image_path(str(row_image_path), self.base_dir or ".")

    def _caption_lookup(self, resolved, raw):
        for k in (resolved, raw, os.path.normpath(str(raw))):
            if k in self.captions_dict:
                return self.captions_dict[k]
        return None

    def __init__(self, df, captions_dict, style_to_idx, clip_preprocess, base_dir=None):
        self.df = df.reset_index(drop=True)
        self.captions_dict = captions_dict
        self.style_to_idx = style_to_idx
        self.clip_preprocess = clip_preprocess
        self.base_dir = base_dir

        self.valid_indices = []
        missing_file = 0
        missing_cap = 0
        for idx in range(len(self.df)):
            row = self.df.iloc[idx]
            raw = str(row["image_path"])
            resolved = self._resolve_path(raw)
            cap = self._caption_lookup(resolved, raw)
            if cap is None:
                missing_cap += 1
                continue
            if not os.path.isfile(resolved):
                missing_file += 1
                continue
            self.valid_indices.append(idx)

        print(
            f"  Dataset: {len(self.valid_indices)} / {len(self.df)} usable | missing caption: {missing_cap} | missing file: {missing_file}"
        )

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        row = self.df.iloc[actual_idx]
        raw_key = str(row["image_path"])
        image_path = self._resolve_path(raw_key)
        if not os.path.isfile(image_path):
            raise FileNotFoundError(f"Missing image after filter: {image_path}")
        image = Image.open(image_path).convert("RGB")
        image = self.clip_preprocess(image)
        caption = self._caption_lookup(image_path, raw_key)
        if caption is None:
            raise RuntimeError(f"Missing caption for {image_path}")
        style = row["style"]
        label = self.style_to_idx[str(style)]
        return {
            "image": image,
            "caption": caption,
            "label": label,
            "style": style,
            "image_path": image_path,
        }


class AttentionFusion(nn.Module):
    def __init__(self, visual_dim, textual_dim, hidden_dim=512):
        super().__init__()
        self.visual_proj = nn.Linear(visual_dim, hidden_dim)
        self.textual_proj = nn.Linear(textual_dim, hidden_dim)
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads=8, batch_first=True)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.final_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, visual_features, textual_features):
        visual_proj = self.visual_proj(visual_features)
        textual_proj = self.textual_proj(textual_features)
        combined = torch.stack([visual_proj, textual_proj], dim=1)
        attended, _ = self.attention(combined, combined, combined)
        attended = self.layer_norm(attended)
        fused = torch.mean(attended, dim=1)
        return self.final_proj(fused)


class MultiModalFashionClassifier(nn.Module):
    def __init__(self, clip_model, fashionbert_model, num_classes, dropout=0.5, visual_dim=512, textual_dim=768):
        super().__init__()
        self.clip_model = clip_model
        self.fashionbert_model = fashionbert_model
        for p in self.clip_model.parameters():
            p.requires_grad = False
        for p in self.fashionbert_model.parameters():
            p.requires_grad = False
        self.fusion = AttentionFusion(visual_dim, textual_dim)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def train(self, mode=True):
        super().train(mode)
        self.clip_model.eval()
        self.fashionbert_model.eval()
        return self

    def forward(self, images, captions):
        with torch.no_grad():
            visual_features = self.clip_model.encode_image(images).float()
            inputs = fashionbert_tokenizer(
                list(captions),
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=128,
            ).to(images.device)
            outputs = self.fashionbert_model(**inputs)
            textual_features = outputs.last_hidden_state[:, 0, :]
        fused = self.fusion(visual_features, textual_features)
        return self.classifier(fused)


def train_epoch(model, train_loader, criterion, optimizer, dev):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for batch in tqdm(train_loader, desc="train", leave=False):
        images = batch["image"].to(dev)
        captions = batch["caption"]
        labels = batch["label"].to(dev)
        optimizer.zero_grad()
        logits = model(images, captions)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pred = logits.argmax(dim=1)
        total += labels.size(0)
        correct += (pred == labels).sum().item()
    acc = correct / max(total, 1)
    return total_loss / max(len(train_loader), 1), acc


def validate_epoch(model, val_loader, criterion, dev, collect_meta=False):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_predictions, all_labels = [], []
    all_paths, all_styles = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="val", leave=False):
            images = batch["image"].to(dev)
            captions = batch["caption"]
            labels = batch["label"].to(dev)
            logits = model(images, captions)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            pred = logits.argmax(dim=1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
            all_predictions.extend(pred.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
            if collect_meta:
                all_paths.extend(list(batch["image_path"]))
                all_styles.extend(list(batch["style"]))
    macro = f1_score(all_labels, all_predictions, average="macro", zero_division=0) if all_predictions else 0.0
    acc = correct / max(total, 1)
    if collect_meta:
        return total_loss / max(len(val_loader), 1), acc, all_predictions, all_labels, macro, all_paths, all_styles
    return total_loss / max(len(val_loader), 1), acc, all_predictions, all_labels, macro


print("✅ Dataset + model classes defined")
ATTN_FUSION_DATASET_CELL_VERSION = 3


✅ Dataset + model classes defined


## Runner: load splits, reload CLIP per seed, train

In [4]:
def load_split_csvs(seed: int):
    split_dir = os.path.join(SPLITS_ROOT, f"seed_{seed}")
    if not os.path.isdir(split_dir):
        raise FileNotFoundError(split_dir)
    assert f"seed_{seed}" in split_dir.replace("\\", "/")
    train_df = pd.read_csv(os.path.join(split_dir, "train.csv"))
    val_df = pd.read_csv(os.path.join(split_dir, "val.csv"))
    test_df = pd.read_csv(os.path.join(split_dir, "test.csv"))
    return split_dir, train_df, val_df, test_df


def make_seed_worker(base_seed):
    def _fn(worker_id):
        w = int(base_seed) + int(worker_id)
        np.random.seed(w)
        torch.manual_seed(w)

    return _fn


def run_robustness_experiment(seed_value, seed_idx):
    _exp_ds_ver = 3
    if globals().get("ATTN_FUSION_DATASET_CELL_VERSION") != _exp_ds_ver:
        raise RuntimeError(
            "Dataset cell is missing or out of date (FashionMultiModalDataset / path remap). "
            "Use **Kernel -> Restart**, then **Run All** from the first cell.\n"
            f"Expected ATTN_FUSION_DATASET_CELL_VERSION == {_exp_ds_ver!r}, "
            f"got {globals().get('ATTN_FUSION_DATASET_CELL_VERSION')!r}."
        )
    print(f"\n{'='*70}\nExperiment {seed_idx}/{len(SEEDS)}: seed {seed_value}\n{'='*70}")

    ckpt_path = Path(CLIP_FINETUNED_ROOT) / f"seed_{seed_value}_best_model.pth"
    assert str(seed_value) in ckpt_path.name

    result_file = os.path.join(METRICS_DIR, "experiments", f"seed_{seed_value}_results.json")
    if os.path.exists(result_file):
        print("  ⏭️  Result exists, skipping")
        with open(result_file) as f:
            return json.load(f)

    split_dir, train_df, val_df, test_df = load_split_csvs(seed_value)

    print("  Split sizes:", len(train_df), len(val_df), len(test_df))
    for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
        ds_tmp = FashionMultiModalDataset(part, captions_dict, style_to_idx, clip_preprocess, base_dir=BASE_DIR)
        print(f"    {name}: with captions+image {len(ds_tmp)} / {len(part)}")

    load_finetuned_clip_weights(clip_model, ckpt_path, map_location=device)

    torch.manual_seed(MODEL_INIT_SEED)
    np.random.seed(MODEL_INIT_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(MODEL_INIT_SEED)

    train_ds = FashionMultiModalDataset(train_df, captions_dict, style_to_idx, clip_preprocess, base_dir=BASE_DIR)
    val_ds = FashionMultiModalDataset(val_df, captions_dict, style_to_idx, clip_preprocess, base_dir=BASE_DIR)
    test_ds = FashionMultiModalDataset(test_df, captions_dict, style_to_idx, clip_preprocess, base_dir=BASE_DIR)

    if len(train_ds) == 0:
        r0 = str(train_df.iloc[0]["image_path"]) if len(train_df) else "<empty train_df>"
        res0 = resolve_split_image_path(r0, BASE_DIR) if len(train_df) else ""
        cap0 = (
            any(k in captions_dict for k in (res0, r0, os.path.normpath(str(r0))))
            if len(train_df)
            else False
        )
        file0 = os.path.isfile(res0) if len(train_df) else False
        if len(train_df) and file0 and cap0:
            raise RuntimeError(
                "Training dataset is empty even though the first train row resolves to an existing file "
                "and has a Qwen caption. The FashionMultiModalDataset class in memory is almost certainly "
                "out of date (old image-path logic).\n\n"
                "Fix: Jupyter menu **Kernel -> Restart**, then **Run All** from the first cell so the "
                "Dataset class and resolve_split_image_path stay in sync.\n"
                f"(Debug: resolved first row -> {res0!r})"
            )
        raise RuntimeError(
            "Training dataset is empty (0 rows with Qwen caption + existing image). Common causes: "
            "stale kernel after notebook edits, wrong PROJECT_ROOT, or caption/path mismatch.\n"
            f"  PROJECT_ROOT={PROJECT_ROOT}\n"
            f"  BASE_DIR={BASE_DIR}\n"
            f"  first train image_path={r0!r}\n"
            f"  resolved={res0!r} exists={os.path.isfile(res0)}\n"
            f"  dataset/ at repo root: {os.path.isdir(os.path.join(BASE_DIR, 'dataset'))}\n"
            f"  data/raw dataset/: {os.path.isdir(os.path.join(BASE_DIR, 'data', 'raw dataset'))}"
        )

    loader_seed = MODEL_INIT_SEED + int(seed_value)
    g = torch.Generator()
    g.manual_seed(loader_seed)
    try:
        train_loader = DataLoader(
            train_ds,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            worker_init_fn=make_seed_worker(loader_seed) if NUM_WORKERS > 0 else None,
            generator=g,
        )
    except ValueError as e:
        if "num_samples" in str(e):
            raise RuntimeError(
                "Training DataLoader failed because the training set has length 0 (RandomSampler). "
                "If you updated this notebook, use **Kernel -> Restart** then **Run All** from the top "
                "so the FashionMultiModalDataset definition matches the setup cell (resolve_split_image_path / "
                "data/raw dataset remap).\n"
                f"len(train_ds)={len(train_ds)!r}\n"
                f"Original error: {e!r}"
            ) from e
        raise
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    train_valid_df = train_ds.df.iloc[train_ds.valid_indices]
    class_weights = compute_class_weight(
        "balanced",
        classes=np.arange(num_classes),
        y=train_valid_df["style"].map(style_to_idx).values,
    )
    class_weights = torch.FloatTensor(class_weights).to(device)

    model = MultiModalFashionClassifier(clip_model, fashionbert_model, num_classes=num_classes, dropout=DROPOUT).to(device)
    n_trainable = sum(p.numel() for p in model.fusion.parameters()) + sum(p.numel() for p in model.classifier.parameters())
    print("  Trainable parameters (fusion + classifier only):", n_trainable)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(
        list(model.fusion.parameters()) + list(model.classifier.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    trainable_names = [n for n, p in model.named_parameters() if p.requires_grad]
    print("  Trainable parameter names (first 25):", trainable_names[:25])
    print("  Total trainable tensors:", len(trainable_names))
    assert all(
        n.startswith("fusion.") or n.startswith("classifier.") for n in trainable_names
    ), f"Unexpected trainable params: {trainable_names}"

    fusion_head_path = os.path.join(ARTIFACTS_DIR, "models", f"seed_{seed_value}_best_fusion_head.pth")

    train_losses, val_losses, train_accs, val_accs, val_macro_f1s, learning_rates = [], [], [], [], [], []
    best_val_macro_f1 = -1.0
    best_epoch = 0
    patience_counter = 0
    best_val_loss = float("inf")
    early_stopped = False
    start_time = time.time()

    for epoch in range(MAX_EPOCHS):
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        va_loss, va_acc, _, _, va_f1 = validate_epoch(model, val_loader, criterion, device)
        scheduler.step()
        learning_rates.append(scheduler.get_last_lr()[0])
        train_losses.append(tr_loss)
        val_losses.append(va_loss)
        train_accs.append(tr_acc)
        val_accs.append(va_acc)
        val_macro_f1s.append(va_f1)

        if va_f1 > best_val_macro_f1:
            best_val_macro_f1 = va_f1
            best_epoch = epoch + 1
            patience_counter = 0
            torch.save(
                {
                    "fusion": model.fusion.state_dict(),
                    "classifier": model.classifier.state_dict(),
                    "seed": int(seed_value),
                    "split_mode": "csv",
                    "clip_checkpoint": str(ckpt_path),
                    "caption_csv": CAPTION_CSV,
                    "best_epoch": int(best_epoch),
                    "best_val_macro_f1": float(best_val_macro_f1),
                },
                fusion_head_path,
            )
        else:
            patience_counter += 1
        if va_loss < best_val_loss:
            best_val_loss = va_loss
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            early_stopped = True
            print(f"  Early stop at epoch {epoch+1}")
            break
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1}: tr_loss={tr_loss:.4f} val_loss={va_loss:.4f} val_macro_f1={va_f1:.4f}")

    total_time = time.time() - start_time
    if not os.path.exists(fusion_head_path):
        torch.save(
            {
                "fusion": model.fusion.state_dict(),
                "classifier": model.classifier.state_dict(),
                "seed": int(seed_value),
                "split_mode": "csv",
                "clip_checkpoint": str(ckpt_path),
                "caption_csv": CAPTION_CSV,
                "best_epoch": int(best_epoch),
                "best_val_macro_f1": float(best_val_macro_f1),
                "fallback_save": True,
            },
            fusion_head_path,
        )
        print("  Warning: no fusion-head checkpoint during training; saved fallback from last weights.")

    ck = torch.load(fusion_head_path, map_location=device)
    model.fusion.load_state_dict(ck["fusion"])
    model.classifier.load_state_dict(ck["classifier"])
    model.eval()
    te_loss, te_acc, te_pred, te_lab, te_f1, te_paths, te_styles = validate_epoch(
        model, test_loader, criterion, device, collect_meta=True
    )

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes[0, 0].plot(train_losses, label="train")
    axes[0, 0].plot(val_losses, label="val")
    axes[0, 0].legend()
    axes[0, 1].plot(train_accs, label="train acc")
    axes[0, 1].plot(val_accs, label="val acc")
    axes[0, 1].set_ylabel("accuracy (0–1)")
    axes[0, 1].legend()
    axes[1, 0].plot(val_macro_f1s, label="val macro F1")
    axes[1, 0].legend()
    axes[1, 1].axis("off")
    axes[1, 1].text(
        0.05,
        0.5,
        f"seed={seed_value}\nbest_val_macro_f1={best_val_macro_f1:.4f}\ntest_macro_f1={te_f1:.4f}\ntest_acc={te_acc:.4f}",
        fontsize=11,
        family="monospace",
    )
    plt.suptitle(
        f"Attention fusion + Qwen — seed {seed_value} (CSV splits; fine-tuned CLIP)"
    )
    plt.tight_layout()
    plt.savefig(os.path.join(ARTIFACTS_DIR, "learning_curves", f"seed_{seed_value}_learning_curves.png"), dpi=200, bbox_inches="tight")
    plt.close()

    results = {
        "experiment_id": f"seed_{seed_value}",
        "seed_value": seed_value,
        "seed_index": seed_idx,
        "timestamp": datetime.now().isoformat(),
        "protocol": {
            "split_mode": "csv",
            "splits": "data/splits/seed_<N>/{train,val,test}.csv",
            "clip_checkpoint": str(ckpt_path),
            "caption_csv": CAPTION_CSV,
            "legacy_table_csv": None,
            "fusion": "attention",
            "fusion_head_checkpoint": fusion_head_path,
        },
        "configuration": {
            "learning_rate": float(LEARNING_RATE),
            "batch_size": BATCH_SIZE,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "dropout": DROPOUT,
            "weight_decay": float(WEIGHT_DECAY),
            "max_epochs": MAX_EPOCHS,
            "model_init_seed": MODEL_INIT_SEED,
        },
        "validation_metrics": {
            "best_val_macro_f1": float(best_val_macro_f1),
            "best_epoch": int(best_epoch),
            "best_val_accuracy": float(val_accs[best_epoch - 1]) if best_epoch > 0 else 0.0,
        },
        "test_metrics": {
            "test_macro_f1": float(te_f1),
            "test_accuracy": float(te_acc),
            "test_loss": float(te_loss),
            "test_predictions": [int(x) for x in te_pred],
            "test_labels": [int(x) for x in te_lab],
            "test_image_paths": [str(x) for x in te_paths],
            "test_styles": [str(x) for x in te_styles],
        },
        "caption_coverage": {
            "train_rows_csv": int(len(train_df)),
            "val_rows_csv": int(len(val_df)),
            "test_rows_csv": int(len(test_df)),
            "train_with_caption_and_file": int(len(train_ds)),
            "val_with_caption_and_file": int(len(val_ds)),
            "test_with_caption_and_file": int(len(test_ds)),
        },
        "training_info": {
            "total_epochs": len(train_losses),
            "early_stopped": early_stopped,
            "total_time_minutes": float(total_time / 60.0),
        },
        "data_split_info": {
            "split_dir": split_dir,
            "train_size": len(train_df),
            "val_size": len(val_df),
            "test_size": len(test_df),
        },
    }
    out_json = os.path.join(METRICS_DIR, "experiments", f"seed_{seed_value}_results.json")
    with open(out_json, "w") as f:
        json.dump(results, f, indent=2)
    print(f"  ✅ Done seed {seed_value} | best val macro F1={best_val_macro_f1:.4f} | test macro F1={te_f1:.4f}")
    return results


if RUN_ALL_SEEDS:
    all_results = []
    failed = []
    for i, sv in enumerate(SEEDS, 1):
        try:
            all_results.append(run_robustness_experiment(sv, i))
        except Exception as e:
            print(f"FAIL seed {sv}: {e}")
            failed.append((sv, str(e)))
    print("Completed:", len(all_results), "Failed:", len(failed))
else:
    for i, sv in enumerate(SMOKE_SEEDS, 1):
        run_robustness_experiment(sv, i)



Experiment 1/30: seed 13
  ⏭️  Result exists, skipping

Experiment 2/30: seed 14
  ⏭️  Result exists, skipping

Experiment 3/30: seed 16
  ⏭️  Result exists, skipping

Experiment 4/30: seed 17
  ⏭️  Result exists, skipping

Experiment 5/30: seed 45
  ⏭️  Result exists, skipping

Experiment 6/30: seed 48
  Split sizes: 9261 1984 1985
  Dataset: 9235 / 9261 usable | missing caption: 26 | missing file: 0
    train: with captions+image 9235 / 9261
  Dataset: 1981 / 1984 usable | missing caption: 3 | missing file: 0
    val: with captions+image 1981 / 1984
  Dataset: 1980 / 1985 usable | missing caption: 5 | missing file: 0
    test: with captions+image 1980 / 1985
  Loaded fine-tuned CLIP: seed_48_best_model.pth | 302 tensor keys
  Dataset: 9235 / 9261 usable | missing caption: 26 | missing file: 0
  Dataset: 1981 / 1984 usable | missing caption: 3 | missing file: 0
  Dataset: 1980 / 1985 usable | missing caption: 5 | missing file: 0
  Trainable parameters (fusion + classifier only): 2136

  Epoch 1: tr_loss=0.9884 val_loss=0.5363 val_macro_f1=0.8564


  Epoch 5: tr_loss=0.0254 val_loss=0.9670 val_macro_f1=0.8562


  Early stop at epoch 7


  ✅ Done seed 48 | best val macro F1=0.8586 | test macro F1=0.8533

Experiment 7/30: seed 53
  Split sizes: 9261 1984 1985
  Dataset: 9238 / 9261 usable | missing caption: 23 | missing file: 0
    train: with captions+image 9238 / 9261
  Dataset: 1981 / 1984 usable | missing caption: 3 | missing file: 0
    val: with captions+image 1981 / 1984
  Dataset: 1977 / 1985 usable | missing caption: 8 | missing file: 0
    test: with captions+image 1977 / 1985
  Loaded fine-tuned CLIP: seed_53_best_model.pth | 302 tensor keys
  Dataset: 9238 / 9261 usable | missing caption: 23 | missing file: 0
  Dataset: 1981 / 1984 usable | missing caption: 3 | missing file: 0
  Dataset: 1977 / 1985 usable | missing caption: 8 | missing file: 0
  Trainable parameters (fusion + classifier only): 2136718
  Trainable parameter names (first 25): ['fusion.visual_proj.weight', 'fusion.visual_proj.bias', 'fusion.textual_proj.weight', 'fusion.textual_proj.bias', 'fusion.attention.in_proj_weight', 'fusion.attention.i

...

  Epoch 1: tr_loss=1.1203 val_loss=0.5071 val_macro_f1=0.8538


  Epoch 5: tr_loss=0.1796 val_loss=0.6433 val_macro_f1=0.8523


  Epoch 10: tr_loss=0.1356 val_loss=0.6743 val_macro_f1=0.8576


  Early stop at epoch 11


  ✅ Done seed 457 | best val macro F1=0.8616 | test macro F1=0.8633
Completed: 30 Failed: 0


In [6]:
# Compares attention fusion vs fine-tuned CLIP **per split seed** from saved metrics JSON,
# then runs paired one-sided tests (H1: fusion > CLIP) on macro F1.

from __future__ import annotations

import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:

    def display(obj):
        print(obj)


# --- resolve PROJECT_ROOT (same pattern as the training cell) ---
_walk = os.path.abspath(os.getcwd())
for _ in range(10):
    if os.path.isdir(os.path.join(_walk, "experiments")) and os.path.isdir(os.path.join(_walk, "data")):
        PROJECT_ROOT = _walk
        break
    _walk = os.path.dirname(_walk)
else:
    PROJECT_ROOT = os.path.abspath(os.getcwd())

try:
    EXPERIMENT_ROOT  # type: ignore[name-defined]
except NameError:
    EXPERIMENT_ROOT = os.path.join(
        PROJECT_ROOT, "experiments", "attention_fusion_finetuned_clip_qwen_v2"
    )

CLIP_METRICS_DIR = os.path.join(
    PROJECT_ROOT,
    "experiments",
    "imageonly_clip_finetuned_robustness",
    "metrics",
    "experiments",
)
FUSION_METRICS_DIR = os.path.join(EXPERIMENT_ROOT, "metrics", "experiments")


def _load_seed_metrics(metrics_dir: str) -> dict[int, dict]:
    out: dict[int, dict] = {}
    d = Path(metrics_dir)
    if not d.is_dir():
        raise FileNotFoundError(f"Metrics dir not found: {metrics_dir}")
    for p in d.glob("seed_*_results.json"):
        m = re.match(r"seed_(\d+)_results\.json$", p.name)
        if not m:
            continue
        seed = int(m.group(1))
        with open(p, "r", encoding="utf-8") as f:
            out[seed] = json.load(f)
    return out


def _extract_metrics(doc: dict, prefix: str) -> dict:
    vm = doc.get("validation_metrics") or {}
    tm = doc.get("test_metrics") or {}
    ti = doc.get("training_info") or {}
    best_epoch = vm.get("best_epoch")
    if best_epoch is None:
        best_epoch = ti.get("best_epoch")
    return {
        f"{prefix}_best_val_macro_f1": vm.get("best_val_macro_f1"),
        f"{prefix}_best_epoch": best_epoch,
        f"{prefix}_test_macro_f1": tm.get("test_macro_f1"),
        f"{prefix}_test_accuracy": tm.get("test_accuracy"),
    }


clip_by_seed = _load_seed_metrics(CLIP_METRICS_DIR)
fusion_by_seed = _load_seed_metrics(FUSION_METRICS_DIR)

common = sorted(set(clip_by_seed) & set(fusion_by_seed))
missing_clip = sorted(set(fusion_by_seed) - set(clip_by_seed))
missing_fusion = sorted(set(clip_by_seed) - set(fusion_by_seed))

if missing_clip:
    print(
        "Seeds with fusion JSON but no CLIP JSON:",
        missing_clip[:20],
        f"... ({len(missing_clip)} total)" if len(missing_clip) > 20 else "",
    )
if missing_fusion:
    print(
        "Seeds with CLIP JSON but no fusion JSON:",
        missing_fusion[:20],
        f"... ({len(missing_fusion)} total)" if len(missing_fusion) > 20 else "",
    )

rows = []
for s in common:
    rows.append({"seed": s, **_extract_metrics(clip_by_seed[s], "clip"), **_extract_metrics(fusion_by_seed[s], "fusion")})

df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError("No overlapping seeds between CLIP and fusion metrics.")

df["delta_val_macro_f1"] = df["fusion_best_val_macro_f1"] - df["clip_best_val_macro_f1"]
df["delta_test_macro_f1"] = df["fusion_test_macro_f1"] - df["clip_test_macro_f1"]

cols = [
    "seed",
    "clip_best_val_macro_f1",
    "fusion_best_val_macro_f1",
    "delta_val_macro_f1",
    "clip_test_macro_f1",
    "fusion_test_macro_f1",
    "delta_test_macro_f1",
    "clip_best_epoch",
    "fusion_best_epoch",
    "clip_test_accuracy",
    "fusion_test_accuracy",
]
df = df[[c for c in cols if c in df.columns]].sort_values("seed").reset_index(drop=True)

print(f"Paired seeds: n={len(df)}")
print("CLIP metrics:", CLIP_METRICS_DIR)
print("Fusion metrics:", FUSION_METRICS_DIR)
display(df)

# --- summary table ---
summary = pd.DataFrame(
    [
        {
            "metric": "best_val_macro_f1",
            "clip_mean": df["clip_best_val_macro_f1"].mean(),
            "fusion_mean": df["fusion_best_val_macro_f1"].mean(),
            "mean_delta_fusion_minus_clip": df["delta_val_macro_f1"].mean(),
            "fusion_wins": int((df["delta_val_macro_f1"] > 0).sum()),
            "ties_abs_le_1e6": int((df["delta_val_macro_f1"].abs() <= 1e-6).sum()),
        },
        {
            "metric": "test_macro_f1",
            "clip_mean": df["clip_test_macro_f1"].mean(),
            "fusion_mean": df["fusion_test_macro_f1"].mean(),
            "mean_delta_fusion_minus_clip": df["delta_test_macro_f1"].mean(),
            "fusion_wins": int((df["delta_test_macro_f1"] > 0).sum()),
            "ties_abs_le_1e6": int((df["delta_test_macro_f1"].abs() <= 1e-6).sum()),
        },
    ]
)
print("\nSummary (paired by seed)")
display(summary)


def paired_tests(a: np.ndarray, b: np.ndarray, label: str) -> pd.DataFrame:
    """H1: mean/median of (a - b) > 0 with a = fusion, b = CLIP."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    diff = a - b
    records: list[dict] = []

    try:
        from scipy import stats
    except ImportError:
        records.append(
            {
                "comparison": label,
                "test": "(skipped)",
                "statistic": np.nan,
                "p_value": np.nan,
                "note": "install scipy for paired t-test and Wilcoxon",
            }
        )
        return pd.DataFrame(records)

    # Paired t-test (one-sided: fusion > CLIP)
    try:
        t_res = stats.ttest_rel(a, b, alternative="greater")
        records.append(
            {
                "comparison": label,
                "test": "paired t-test (greater)",
                "statistic": float(t_res.statistic),
                "p_value": float(t_res.pvalue),
                "note": "H1: mean(fusion - CLIP) > 0",
            }
        )
    except TypeError:
        t_res = stats.ttest_rel(a, b)
        p_two = float(t_res.pvalue)
        mean_d = float(np.mean(diff))
        p_one = p_two / 2.0 if mean_d > 0 else 1.0 - p_two / 2.0
        records.append(
            {
                "comparison": label,
                "test": "paired t-test (greater, legacy scipy)",
                "statistic": float(t_res.statistic),
                "p_value": p_one,
                "note": "approx one-sided from two-sided p",
            }
        )

    # Wilcoxon signed-rank on differences
    if np.allclose(diff, 0.0, atol=1e-12):
        records.append(
            {
                "comparison": label,
                "test": "Wilcoxon signed-rank (greater)",
                "statistic": np.nan,
                "p_value": 1.0,
                "note": "all paired differences ~ 0",
            }
        )
    else:
        try:
            w_res = stats.wilcoxon(diff, alternative="greater", zero_method="wilcox", mode="auto")
            records.append(
                {
                    "comparison": label,
                    "test": "Wilcoxon signed-rank (greater)",
                    "statistic": float(w_res.statistic),
                    "p_value": float(w_res.pvalue),
                    "note": "H1: median(fusion - CLIP) > 0",
                }
            )
        except TypeError:
            w_res = stats.wilcoxon(diff, zero_method="wilcox")
            p_two = float(w_res.pvalue)
            med_d = float(np.median(diff))
            p_one = p_two / 2.0 if med_d > 0 else 1.0 - p_two / 2.0
            records.append(
                {
                    "comparison": label,
                    "test": "Wilcoxon (greater, legacy scipy)",
                    "statistic": float(w_res.statistic),
                    "p_value": p_one,
                    "note": "approx one-sided from two-sided p",
                }
            )

    return pd.DataFrame(records)


tests_df = pd.concat(
    [
        paired_tests(df["fusion_test_macro_f1"].values, df["clip_test_macro_f1"].values, "test_macro_f1"),
        paired_tests(
            df["fusion_best_val_macro_f1"].values,
            df["clip_best_val_macro_f1"].values,
            "best_val_macro_f1",
        ),
    ],
    ignore_index=True,
)

alpha = 0.05
tests_df["significant_at_0.05"] = tests_df["p_value"] < alpha
print("\nPaired one-sided tests (fusion > CLIP). Interpret with caution: seeds are not i.i.d. if splits overlap protocol-wise, but paired deltas are still descriptive.")
display(tests_df)
# --- Paired tests (same pattern as TextOnly_Qwen_StratifiedSplits_Robustness): t, Wilcoxon, bootstrap ---
from scipy import stats

if "df" not in dir():
    raise RuntimeError("Run the cell that builds `df` first.")

ALPHA = 0.05
N_BOOT = 10_000
RNG = np.random.default_rng(42)


def bootstrap_paired_mean_diff(d: np.ndarray, n_boot: int = N_BOOT, rng: np.random.Generator = RNG):
    """Resample seed indices with replacement; mean of d on each replicate."""
    d = np.asarray(d, dtype=float)
    n = len(d)
    if n < 2:
        raise ValueError("Need at least 2 pairs for bootstrap.")
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_means = d[idx].mean(axis=1)
    ci_lo, ci_hi = np.quantile(boot_means, [0.025, 0.975])
    # One-sided p (fusion > CLIP): small if mass of bootstrap means is above 0
    p_one_sided = (1.0 + np.sum(boot_means <= 0.0)) / (n_boot + 1.0)
    return float(ci_lo), float(ci_hi), float(p_one_sided), boot_means


def paired_test_rows(metric_label: str, fusion: np.ndarray, clip: np.ndarray, alpha: float = ALPHA):
    """Paired t (greater), Wilcoxon on differences (greater), parametric 95% CI on mean(d), bootstrap CI + boot p."""
    fusion = np.asarray(fusion, dtype=float)
    clip = np.asarray(clip, dtype=float)
    d = fusion - clip
    n = len(d)
    mean_d = float(np.mean(d))
    std_d = float(np.std(d, ddof=1)) if n > 1 else 0.0
    cohen_dz = mean_d / std_d if std_d > 1e-12 else np.nan

    try:
        t_res = stats.ttest_rel(fusion, clip, alternative="greater")
        t_stat = float(t_res.statistic)
        t_p_one = float(t_res.pvalue)
    except TypeError:
        t_res = stats.ttest_rel(fusion, clip)
        t_stat = float(t_res.statistic)
        p_two = float(t_res.pvalue)
        t_p_one = p_two / 2.0 if mean_d > 0 else 1.0 - p_two / 2.0

    if np.allclose(d, 0.0, atol=1e-12):
        w_stat = np.nan
        w_p_one = 1.0
    else:
        try:
            w_res = stats.wilcoxon(d, alternative="greater", zero_method="wilcox", method="approx")
        except (TypeError, ValueError):
            try:
                w_res = stats.wilcoxon(d, alternative="greater", zero_method="wilcox", mode="auto")
            except (TypeError, ValueError):
                w_res = stats.wilcoxon(d, alternative="greater", zero_method="wilcox")
        w_stat = float(w_res.statistic) if w_res.statistic is not None else np.nan
        w_p_one = float(w_res.pvalue)

    if n > 1:
        se = stats.sem(d)
        h = se * stats.t.ppf((1 + 0.95) / 2.0, n - 1)
        ci_lo, ci_hi = mean_d - h, mean_d + h
    else:
        ci_lo = ci_hi = mean_d

    wins = int(np.sum(d > 0))
    ties = int(np.sum(np.isclose(d, 0.0, atol=1e-12)))
    losses = int(np.sum(d < 0))

    boot_ci_lo, boot_ci_hi, boot_p_one, _ = bootstrap_paired_mean_diff(d, n_boot=N_BOOT, rng=RNG)
    boot_ci_entirely_gt_0 = bool(boot_ci_lo > 0.0)

    return {
        "metric": metric_label,
        "n_pairs": n,
        "mean_diff_fusion_minus_clip": mean_d,
        "std_diff": std_d,
        "cohen_dz_paired": cohen_dz,
        "ci95_mean_diff_lo": ci_lo,
        "ci95_mean_diff_hi": ci_hi,
        "t_statistic": t_stat,
        "t_p_one_sided_fusion_gt_clip": t_p_one,
        "reject_t_H0_at_alpha": bool(t_p_one < alpha),
        "wilcoxon_statistic": w_stat,
        "wilcoxon_p_one_sided_fusion_gt_clip": w_p_one,
        "reject_wilcoxon_H0_at_alpha": bool(w_p_one < alpha),
        "wins_fusion": wins,
        "ties": ties,
        "losses_fusion": losses,
        "boot_ci95_mean_diff_lo": boot_ci_lo,
        "boot_ci95_mean_diff_hi": boot_ci_hi,
        "boot_ci_entirely_gt_0": boot_ci_entirely_gt_0,
        "boot_p_one_sided_fusion_gt_clip": boot_p_one,
        "reject_boot_H0_at_alpha": bool(boot_p_one < alpha),
    }


df_paired_tests = pd.DataFrame(
    [
        paired_test_rows("test_macro_f1", df["fusion_test_macro_f1"].values, df["clip_test_macro_f1"].values),
        paired_test_rows(
            "best_val_macro_f1",
            df["fusion_best_val_macro_f1"].values,
            df["clip_best_val_macro_f1"].values,
        ),
    ]
)

print(
    f"Paired tests: one-sided fusion > CLIP | alpha = {ALPHA} | "
    f"bootstrap mean-delta: {N_BOOT:,} resamples, RNG seed=42"
)
print(
    "Parametric CI = classical t interval on mean paired difference; "
    "Boot CI = percentile interval on bootstrap mean(d); "
    "boot p = (1 + #{boot mean <= 0}) / (n_boot+1)."
)
display(df_paired_tests)

Paired seeds: n=30
CLIP metrics: /home/ding-zhang/Dongmei/DATA255/Project/experiments/imageonly_clip_finetuned_robustness/metrics/experiments
Fusion metrics: /home/ding-zhang/Dongmei/DATA255/Project/experiments/attention_fusion_finetuned_clip_qwen_v2/metrics/experiments


,seed,clip_best_val_macro_f1,fusion_best_val_macro_f1,delta_val_macro_f1,clip_test_macro_f1,fusion_test_macro_f1,delta_test_macro_f1,clip_best_epoch,fusion_best_epoch,clip_test_accuracy,fusion_test_accuracy
0,13,0.864177,0.861274,-0.002903,0.856459,0.855748,-0.000711,7,9,85.598787,0.854977
1,14,0.868878,0.864121,-0.004757,0.844758,0.843582,-0.001176,12,4,84.460141,0.841574
2,16,0.846987,0.849017,0.002030,0.839231,0.844222,0.004991,13,7,83.854692,0.842583
3,17,0.852015,0.856309,0.004294,0.842217,0.845202,0.002986,6,3,84.335523,0.844871
4,45,0.868400,0.864450,-0.003950,0.851487,0.853068,0.001580,10,5,84.883721,0.850859
5,48,0.860972,0.858607,-0.002366,0.860307,0.853300,-0.007007,12,2,85.959596,0.852525
6,53,0.840287,0.841864,0.001577,0.858126,0.856940,-0.001186,6,2,85.685382,0.856854
7,58,0.851648,0.850954,-0.000694,0.862055,0.858039,-0.004016,10,1,86.147624,0.857432
8,72,0.849684,0.856333,0.006648,0.842042,0.845009,0.002967,8,2,84.367121,0.846193
9,102,0.863030,0.865423,0.002393,0.867332,0.860083,-0.007249,16,3,86.686838,0.859304



Summary (paired by seed)


,metric,clip_mean,fusion_mean,mean_delta_fusion_minus_clip,fusion_wins,ties_abs_le_1e6
0,best_val_macro_f1,0.856078,0.856944,0.000867,18,0
1,test_macro_f1,0.850517,0.851191,0.000674,16,0



Paired one-sided tests (fusion > CLIP). Interpret with caution: seeds are not i.i.d. if splits overlap protocol-wise, but paired deltas are still descriptive.


,comparison,test,statistic,p_value,note,significant_at_0.05
0,test_macro_f1,paired t-test (greater),0.764926,0.225248,H1: mean(fusion - CLIP) > 0,False
1,test_macro_f1,Wilcoxon signed-rank (greater),266.000000,0.251381,H1: median(fusion - CLIP) > 0,False
2,best_val_macro_f1,paired t-test (greater),1.457798,0.077821,H1: mean(fusion - CLIP) > 0,False
3,best_val_macro_f1,Wilcoxon signed-rank (greater),292.000000,0.114276,H1: median(fusion - CLIP) > 0,False


Paired tests: one-sided fusion > CLIP | alpha = 0.05 | bootstrap mean-delta: 10,000 resamples, RNG seed=42
Parametric CI = classical t interval on mean paired difference; Boot CI = percentile interval on bootstrap mean(d); boot p = (1 + #{boot mean <= 0}) / (n_boot+1).


,metric,n_pairs,mean_diff_fusion_minus_clip,std_diff,cohen_dz_paired,ci95_mean_diff_lo,ci95_mean_diff_hi,t_statistic,t_p_one_sided_fusion_gt_clip,reject_t_H0_at_alpha,...,wilcoxon_p_one_sided_fusion_gt_clip,reject_wilcoxon_H0_at_alpha,wins_fusion,ties,losses_fusion,boot_ci95_mean_diff_lo,boot_ci95_mean_diff_hi,boot_ci_entirely_gt_0,boot_p_one_sided_fusion_gt_clip,reject_boot_H0_at_alpha
0,test_macro_f1,30,0.000674,0.004828,0.139656,-0.001129,0.002477,0.764926,0.225248,False,...,0.245399,False,16,0,14,-0.001048,0.002332,False,0.217478,False
1,best_val_macro_f1,30,0.000867,0.003257,0.266156,-0.000349,0.002083,1.457798,0.077821,False,...,0.110511,False,18,0,12,-0.000254,0.002026,False,0.065593,False
